# 02 — Data Preparation & Baseline Classification
Notebook ini adalah eksperimen awal untuk memprediksi **label kondisi tercatat**. Hasilnya tidak boleh dipakai sebagai keputusan operasional tanpa validasi ahli dan data tambahan.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             balanced_accuracy_score, f1_score, recall_score,
                             precision_score, accuracy_score)

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / "data" / "processed" / "kendaraan_2025_sanitized.csv")
df = df.dropna(subset=["Kondisi"]).copy()
y = df["Kondisi"].astype(str)
# Exclude constant period and identifiers; avoid direct identifier leakage.
X = df.drop(columns=["Kondisi", "periode_data"], errors="ignore")
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Fitur kategorikal:", cat_cols)
print("Fitur numerik:", num_cols)

In [ ]:
preprocessor = ColumnTransformer([
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols),
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_cols)
], remainder="drop")

models = {
    "Dummy most_frequent": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced",
        min_samples_leaf=2, n_jobs=-1
    )
}
results = []
fitted = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = (pipe, pred)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
        "recall_RUSAK": recall_score(y_test, pred, pos_label="RUSAK", zero_division=0),
        "precision_RUSAK": precision_score(y_test, pred, pos_label="RUSAK", zero_division=0)
    })
metrics = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
display(metrics.round(4))

In [ ]:
best_name = metrics.iloc[0]["model"]
best_pipe, best_pred = fitted[best_name]
print("Model dengan macro-F1 tertinggi pada satu holdout:", best_name)
print(classification_report(y_test, best_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, best_pred, labels=["BAIK", "RUSAK"], values_format="d")
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

## Catatan evaluasi

1. Hasil holdout tunggal memiliki ketidakpastian; tambahkan cross-validation pada tahap lanjutan.
2. Periksa kemungkinan kebocoran target melalui `penempatan`, `fungsi`, atau kategori administratif yang mungkin merefleksikan proses pencatatan.
3. `recall_RUSAK` penting karena false negative dapat membuat entri RUSAK terklasifikasi BAIK.
4. Model tidak mengestimasi kesiapan real-time, biaya perbaikan, atau alokasi optimal.
5. Jangan gunakan oversampling sebelum pemisahan data; bila diperlukan, terapkan hanya di pipeline data latih.